# ParkSense AI — Exploratory Data Analysis & Data Cleaning
**Owner**: Eric (backend-ml-analysis)

This notebook cleans the raw 298k Bengaluru violation dataset, converts timestamps to local Indian Standard Time (IST) to align temporal features with local traffic peaks, and exports the processed Parquet file for model training.

In [ ]:
import os
import pandas as pd
import numpy as np
import h3

# Paths
RAW_CSV_PATH = "../data/jan to may police violation_anonymized791b166.csv"
PROCESSED_DIR = "../data/processed"
OUTPUT_PARQUET_PATH = os.path.join(PROCESSED_DIR, "violations_clean.parquet")

## 1. Load Data & Inspect Quality
We read the raw 110MB CSV dataset containing Bengaluru police violations.

In [ ]:
print(f"Loading raw CSV from {RAW_CSV_PATH}...")
df = pd.read_csv(RAW_CSV_PATH)
print(f"Loaded {len(df):,} rows.")

print("\n--- Missing Value Count (Null Analysis) ---")
print(df.isnull().sum())

## 2. Spatial & Temporal Cleaning
1. **Coordinates**: Drop coordinates outside Bengaluru boundary box (lat 12.0 to 14.0, lng 77.0 to 79.0) or null coordinates.
2. **Timezone**: Convert datetime from UTC to Indian Standard Time (IST - Asia/Kolkata) so that the temporal analysis matches local traffic rush hours.

In [ ]:
print("Cleaning coordinates...")
df = df.dropna(subset=["latitude", "longitude"])
df = df[(df["latitude"] > 12.0) & (df["latitude"] < 14.0)]
df = df[(df["longitude"] > 77.0) & (df["longitude"] < 79.0)]

print("Converting timestamps to Asia/Kolkata (IST)...")
df["timestamp"] = pd.to_datetime(df["created_datetime"], format="mixed")
df["timestamp"] = df["timestamp"].dt.tz_convert("Asia/Kolkata")
df.drop(columns=["created_datetime"], inplace=True)

## 3. Feature Generation (H3 Spatials & Baseline CIS)
1. **H3 Spatial Bins**: Compute Uber H3 index at resolution 8 to aggregate violations geographically.
2. **Blockage Factor**: Map vehicle types to standard traffic footprint coefficients.
3. **CIS Score**: Compute baseline CIS score based on vehicle size and rush hour indicators.

In [ ]:
print("Computing H3 spatial bins...")
df["h3_cell"] = [h3.latlng_to_cell(lat, lng, 8) for lat, lng in zip(df["latitude"], df["longitude"])]

print("Mapping vehicle blockage factors...")
blockage_map = {
    "CAR": 1.5,
    "MAXI-CAB": 1.6,
    "SCOOTER": 0.5,
    "MOTOR CYCLE": 0.5,
    "THREE WHEELER": 1.0,
    "TEMPO": 1.8,
    "TRUCK": 2.2,
    "BUS": 2.5,
    "TRACTOR": 2.0
}
df["vehicle_type"] = df["vehicle_type"].fillna("UNKNOWN").str.upper()
df["blockage_factor"] = df["vehicle_type"].map(blockage_map).fillna(1.0)

df["hour"] = df["timestamp"].dt.hour
df["temporal_weight"] = np.where(df["hour"].isin([8, 9, 10, 17, 18, 19, 20]), 1.3, 1.0)

print("Computing baseline CIS scores...")
df["cis_score"] = (df["blockage_factor"] * df["temporal_weight"].astype(float) * 30.0).round(2)
df["cis_score"] = df["cis_score"].clip(upper=100.0)

df["validation_status"] = df["validation_status"].fillna("pending").str.lower()
df["vehicle_number"] = df["vehicle_number"].fillna("UNKNOWN").str.upper().str.replace("-", "")

if "id" in df.columns:
    df.rename(columns={"id": "violation_id"}, inplace=True)

# Save required subset of columns
required_cols = [
    "violation_id", "vehicle_number", "vehicle_type", "violation_type",
    "latitude", "longitude", "timestamp", "police_station", "junction_name",
    "validation_status", "cis_score", "h3_cell"
]
df = df[[c for c in required_cols if c in df.columns]]

## 4. Export Cleaned Dataset
Write the final processed output to a lightweight Parquet file.

In [ ]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Saving cleaned dataset to {OUTPUT_PARQUET_PATH}...")
df.to_parquet(OUTPUT_PARQUET_PATH, index=False)
print(f"Successfully saved {len(df):,} rows.")

## 5. Exploratory Data Analysis & Insights
Let's show basic statistics and insights from the processed dataset.

In [ ]:
print("\n--- Top 10 Vehicle Types in Violations ---")
print(df["vehicle_type"].value_counts().head(10))

print("\n--- Peak Hours of Violations (IST) ---")
df_hour = df["timestamp"].dt.hour.value_counts().sort_index()
for hr, count in df_hour.items():
    print(f"{hr:02d}:00 -> {count:,} violations")